# CodeExample 2

This second code example is essentially just a walk-through of a small code snippet. In a short 2-hr tutorial we haven't enough time to explain all the mathematics of transformers and talk you through a fully working codebase for training a decoder-only transformer model and code for using it to do inference (prompt completion). 

Instead the goal here is to show you how easy it is to code one of the main sub-layers of a decoder-transformer-block, namely the multi-headed-self-attention sub-layer. We show how easy it is to do oncee you understand at a high-level the components that make up the decoder-block and when you have access to the most frequently used operations, such as softmax and various tensor manipulations, that are available in a package such as PyTorch which is designed to support construction of deep learning model architectures. The code snippet below shows code for a class defining a self-attention head object.

In [1]:
import torch.nn as nn

class SelfAttentionHead(nn.Module):
    """
    Class definition for a single self-attention head. The attention head transforms 
    the input embedding vector using transformation matrices to create context-aware 
    embedding vectors.
    """
    def __init__(self, d, d_h):
        super().__init__()
        # Initialize transformation matrices for queries, keys, and values
        self.U_Q = nn.Parameter(torch.rand(d, d_h))
        self.U_K = nn.Parameter(torch.rand(d, d_h))
        self.U_V = nn.Parameter(torch.rand(d, d_h))
        self.d_h = d_h  # Dimensionality of vectors in each attention head

    def forward(self, x, mask):
        # 1. Here we transform the input embedding vectors x using the 
        # transformation matrices U_Q, U_K and U_V. This gives us the 
        # query vectors, the key vectors, and the value vectors.
        
        # 2. Note that here the value vectors V are computed as a linear 
        # transformation of the input vectors x, as we mentioned in the lecture notes, 
        # rather than being learnt as independent vectors. The transformation
        # matrix U_V is considered as just additional model parameters that we learn during training.
        # The dimension of the transformation matrix U_V is the as the dimensions 
        # of the other transformation matrices U_Q and U_K, i.e. d x d_h
        Q = x @ self.U_Q
        K = x @ self.U_K
        V = x @ self.U_V

        # This is where we are computing inner products between the query vectors
        # and the key vectors. Note that we explicitly included the square-root factor
        # in the demoninator that we spoke about tin the lecture notes. This is to 
        # ensure the variance of the inner products values is an O(1) quantity so that we 
        # have a stable training process. It is explicitly needed here because we are 
        # talking about implementation, whilst in the mathematics we could absorb it in 
        # the definition of the matrices U_Q and U_K.
        inner_products = Q @ K.transpose(-2, -1) / math.sqrt(self.d_h)

        # Now apply any masking we want. This is where we set the query-key inner 
        # product value to -infinity for any attention matrix elements
        # we want masked out.
        masked_inner_products = inner_products.masked_fill(mask == 0, float("-inf"))

        # Now we convert the inner product values to probabilities by 
        # applying the softmax function to each row of inner-product values.
        attention_weights = torch.softmax(masked_inner_products, dim=-1)

        # Finally take the weighted average of the value vectors V using 
        # the probability weights we just got back from the softmax function
        # and return those weighted average vectors
        return attention_weights @ V